In [1]:
import cv2
import os
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

# ==============================
# PATHS
# ==============================
dataset_path = r"D:\IPCV\LAB-EXP-16\dataset"
images_folder = r"D:\IPCV\LAB-EXP-16\images"
result_folder = r"D:\IPCV\LAB-EXP-16\resultant_img"

os.makedirs(images_folder, exist_ok=True)
os.makedirs(result_folder, exist_ok=True)

# ==============================
# FACE DETECTOR
# ==============================
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

faces = []
labels = []
label_map = {}
current_label = 0

print("📂 Loading dataset...")

# ==============================
# LOAD DATA
# ==============================
for person_name in os.listdir(dataset_path):
    person_path = os.path.join(dataset_path, person_name)

    if not os.path.isdir(person_path):
        continue

    label_map[current_label] = person_name

    for img_name in os.listdir(person_path):

        if not img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue

        img_path = os.path.join(person_path, img_name)
        print(f"Processing: {img_path}")

        img = cv2.imread(img_path)

        if img is None:
            print("❌ Could not read image")
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # ✅ Improved detection settings
        detected_faces = face_cascade.detectMultiScale(gray, 1.1, 3)

        print("Faces found:", len(detected_faces))

        for (x, y, w, h) in detected_faces:
            face = gray[y:y+h, x:x+w]
            face = cv2.resize(face, (100, 100))

            faces.append(face.flatten())
            labels.append(current_label)

    current_label += 1

faces = np.array(faces)
labels = np.array(labels)

# ==============================
# CHECK DATA
# ==============================
print("Total faces collected:", len(faces))

if len(faces) == 0:
    print("❌ No faces detected. Try clearer images.")
    exit()

# ==============================
# TRAIN MODEL
# ==============================
model = KNeighborsClassifier(n_neighbors=3)
model.fit(faces, labels)

print("✅ Model trained!")

# ==============================
# TEST + SAVE OUTPUT
# ==============================
print("🔍 Recognizing...")

test_images = []

for person_name in os.listdir(dataset_path):
    person_path = os.path.join(dataset_path, person_name)

    for img_name in os.listdir(person_path):
        if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            test_images.append(os.path.join(person_path, img_name))

for idx, img_path in enumerate(test_images[:5]):

    img = cv2.imread(img_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    faces_detected = face_cascade.detectMultiScale(gray, 1.1, 3)

    for (x, y, w, h) in faces_detected:
        face = gray[y:y+h, x:x+w]
        face = cv2.resize(face, (100, 100)).flatten().reshape(1, -1)

        pred = model.predict(face)[0]
        name = label_map[pred]

        cv2.rectangle(img, (x, y), (x+w, y+h), (0,255,0), 2)
        cv2.putText(img, name, (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)

    # Save images
    cv2.imwrite(os.path.join(images_folder, f"input_{idx}.png"), img)
    cv2.imwrite(os.path.join(result_folder, f"output_{idx}.png"), img)

print("✅ Done! Check your folders.")

📂 Loading dataset...
Processing: D:\IPCV\LAB-EXP-16\dataset\person1\1.jpeg
Faces found: 0
Processing: D:\IPCV\LAB-EXP-16\dataset\person1\2.jpeg
Faces found: 0
Processing: D:\IPCV\LAB-EXP-16\dataset\person1\3.jpeg
Faces found: 0
Processing: D:\IPCV\LAB-EXP-16\dataset\person1\4.jpeg
Faces found: 0
Processing: D:\IPCV\LAB-EXP-16\dataset\person1\5.jpeg
Faces found: 0
Processing: D:\IPCV\LAB-EXP-16\dataset\person2\1.jpg
Faces found: 1
Processing: D:\IPCV\LAB-EXP-16\dataset\person2\2.jpg
Faces found: 1
Processing: D:\IPCV\LAB-EXP-16\dataset\person2\3.jpg
Faces found: 1
Processing: D:\IPCV\LAB-EXP-16\dataset\person2\4.jpg
Faces found: 1
Processing: D:\IPCV\LAB-EXP-16\dataset\person2\5.jpg
Faces found: 1
Total faces collected: 5
✅ Model trained!
🔍 Recognizing...
✅ Done! Check your folders.


In [2]:
import cv2
import os
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

# ==============================
# 📁 PATHS
# ==============================
dataset_path = r"D:\IPCV\LAB-EXP-16\dataset"
images_folder = r"D:\IPCV\LAB-EXP-16\images"
result_folder = r"D:\IPCV\LAB-EXP-16\resultant_img"

os.makedirs(images_folder, exist_ok=True)
os.makedirs(result_folder, exist_ok=True)

# ==============================
# 📊 LOAD DATA (NO FACE DETECTION)
# ==============================
faces = []
labels = []
label_map = {}
current_label = 0

print("📂 Loading dataset...")

for person_name in os.listdir(dataset_path):
    person_path = os.path.join(dataset_path, person_name)

    if not os.path.isdir(person_path):
        continue

    label_map[current_label] = person_name

    for img_name in os.listdir(person_path):

        if not img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue

        img_path = os.path.join(person_path, img_name)
        print(f"Processing: {img_path}")

        img = cv2.imread(img_path)

        if img is None:
            print("❌ Skipping invalid image")
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # ✅ DIRECTLY USE IMAGE (NO FACE DETECTION)
        face = cv2.resize(gray, (100, 100))

        faces.append(face.flatten())
        labels.append(current_label)

    current_label += 1

faces = np.array(faces)
labels = np.array(labels)

print("Total samples:", len(faces))

if len(faces) == 0:
    print("❌ No data found.")
    exit()

# ==============================
# 🤖 TRAIN MODEL
# ==============================
model = KNeighborsClassifier(n_neighbors=3)
model.fit(faces, labels)

print("✅ Model trained!")

# ==============================
# 🔍 TEST + RECOGNITION
# ==============================
print("🔍 Recognizing...")

test_images = []

for person_name in os.listdir(dataset_path):
    person_path = os.path.join(dataset_path, person_name)

    for img_name in os.listdir(person_path):
        if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            test_images.append(os.path.join(person_path, img_name))

# ==============================
# 💾 SAVE OUTPUT
# ==============================
for idx, img_path in enumerate(test_images[:5]):

    img = cv2.imread(img_path)

    if img is None:
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    face = cv2.resize(gray, (100, 100)).flatten().reshape(1, -1)

    pred = model.predict(face)[0]
    name = label_map[pred]

    # ✅ ONLY TEXT (NO BOUNDING BOX)
    cv2.putText(img, f"Predicted: {name}",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 255, 0),
                2)

    # Save images
    cv2.imwrite(os.path.join(images_folder, f"input_{idx}.png"), img)
    cv2.imwrite(os.path.join(result_folder, f"output_{idx}.png"), img)

print("✅ Done! Check your folders.")

📂 Loading dataset...
Processing: D:\IPCV\LAB-EXP-16\dataset\person1\1.jpeg
Processing: D:\IPCV\LAB-EXP-16\dataset\person1\2.jpeg
Processing: D:\IPCV\LAB-EXP-16\dataset\person1\3.jpeg
Processing: D:\IPCV\LAB-EXP-16\dataset\person1\4.jpeg
Processing: D:\IPCV\LAB-EXP-16\dataset\person1\5.jpeg
Processing: D:\IPCV\LAB-EXP-16\dataset\person2\1.jpg
Processing: D:\IPCV\LAB-EXP-16\dataset\person2\2.jpg
Processing: D:\IPCV\LAB-EXP-16\dataset\person2\3.jpg
Processing: D:\IPCV\LAB-EXP-16\dataset\person2\4.jpg
Processing: D:\IPCV\LAB-EXP-16\dataset\person2\5.jpg
Total samples: 10
✅ Model trained!
🔍 Recognizing...
✅ Done! Check your folders.
